---

## Step 1: Install the AgentCore CLI

In [ ]:
!uv pip install --system bedrock-agentcore-starter-toolkit bedrock-agentcore strands-agents boto3

---

## Step 2: Set up execution roles

Creates the IAM execution roles for the AgentCore runtimes (idempotent — safe to re-run).

In [ ]:
import boto3, json, os

# Set up execution roles for AgentCore runtimes.
# Creates them if missing; uses them if already exist (idempotent).

iam = boto3.client("iam")
sts = boto3.client("sts")
account = sts.get_caller_identity()["Account"]
region  = os.environ.get("AWS_REGION", "us-east-1")
bucket  = f"bedrock-agentcore-deploy-{account}-{region}"

trust = json.dumps({
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow",
                   "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
                   "Action": "sts:AssumeRole"}]
})

for role_name, env_var in [
    ("workshop-agentcore-m4-runtime-role",      "AGENTCORE_RUNTIME_ROLE_ARN"),
    ("workshop-agentcore-m4-orchestrator-role", "AGENTCORE_ORCHESTRATOR_ROLE_ARN"),
]:
    try:
        arn = iam.get_role(RoleName=role_name)["Role"]["Arn"]
        print(f"  {role_name}")
        print(f"  {arn}\n")
    except iam.exceptions.NoSuchEntityException:
        try:
            arn = iam.create_role(RoleName=role_name,
                                  AssumeRolePolicyDocument=trust,
                                  Description="AgentCore runtime role")["Role"]["Arn"]
            print(f"  Created: {role_name}")
            print(f"  {arn}\n")
        except Exception as e:
            print(f"  Could not create {role_name}: {e}")
            print(f"  Set {env_var} env var to a pre-existing role ARN\n")
            arn = None
    except Exception as e:
        print(f"  {role_name}: {e}\n")
        arn = None

    if arn:
        os.environ[env_var] = arn

---

## Step 3: Deploy

Runs all four runtimes (~3-5 min). The cell below executes `deploy.py` directly so it picks up the role ARNs set in the previous cell.

In [ ]:
import subprocess, sys, re, os

proc = subprocess.Popen(
    [sys.executable, "deploy.py", "--name-prefix", "m4"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env={**os.environ},
)
lines = []
for line in proc.stdout:
    print(line, end="", flush=True)
    lines.append(line)
proc.wait()

if proc.returncode != 0:
    raise SystemExit(f"deploy.py failed (exit {proc.returncode})")

m = re.search(r"Orchestrator ARN:\s*(arn:aws:[^\s]+)", "".join(lines))
RUNTIME_ARN = m.group(1) if m else ""
if RUNTIME_ARN:
    print(f"\nRUNTIME_ARN = {RUNTIME_ARN}")
else:
    print("\nWARNING: could not extract RUNTIME_ARN from output. Run cleanup and redeploy.")

---

## Step 4: Invoke the deployed agent

The `RUNTIME_ARN` was captured from the deploy output above. Run the cell below to invoke the orchestrator.

In [ ]:
import os, boto3, json, uuid
from botocore.config import Config

REGION = os.environ.get("AWS_REGION", "us-east-1")

if not RUNTIME_ARN:
    raise ValueError("RUNTIME_ARN not set — re-run Step 2 (Deploy).")

client = boto3.client(
    "bedrock-agentcore",
    region_name=REGION,
    config=Config(read_timeout=300),
)

response = client.invoke_agent_runtime(
    agentRuntimeArn=RUNTIME_ARN,
    runtimeSessionId=str(uuid.uuid4()),
    payload=json.dumps({
        "prompt": "NovaCart Premium Tier: Options A ($19.99/mo invite-only), B ($14.99/mo 5% pilot), C ($12.99/mo full launch). Target: +15% CLV in 6 months."
    }).encode(),
    qualifier="DEFAULT",
)

result = json.loads(response["response"].read())
print(result.get("response", result))

---

## Step 5: Observability

After invoking, traces appear in **CloudWatch > X-Ray > Traces** or **Amazon Bedrock > AgentCore > Observability**.

What you see per invocation:
- Root span per `invoke_agent_runtime` call
- Child span per `Agent()` call inside the pipeline
- Tool call spans nested under each agent
- Duration breakdown per stage

No extra configuration needed: `aws-opentelemetry-distro` is in `requirements.txt` and AgentCore installs it automatically.

---

## Step 6: Cleanup

Uncomment and run the cell below to delete all AWS resources created by this module.

In [ ]:
import subprocess, sys, os

# Uncomment the lines below and run this cell to delete all resources created by this module.
# This is irreversible — confirm the runtimes are no longer needed before running.

# subprocess.run(
#     [sys.executable, "cleanup.py", "--name-prefix", "m4",
#     ],
#     env={**os.environ},
#     check=True,
# )

# Verify after cleanup:
# import boto3
# REGION = os.environ.get("AWS_REGION", "us-east-1")
# remaining = boto3.client("bedrock-agentcore-control", region_name=REGION) \
#     .list_agent_runtimes().get("agentRuntimes", [])
# print([rt["agentRuntimeName"] for rt in remaining] or "no runtimes remaining")